# Backprop practice 4: forward and backward by hand (PyTorch)

Four networks in the style of practice 2 and 3. For each one, write `forward` with torch ops (store what backward
needs) and `backward` **by hand** (no `.backward()`, no `torch.autograd`), returning a dict of gradients keyed by
parameter name, e.g. `{"W1": dW1, "gamma": dgamma}`.

Each test checks your forward loss against a fixed value, then runs autograd on **your** forward and compares
every gradient with yours. float64 throughout. Write the shape next to every line.

In [ ]:
import torch

torch.set_default_dtype(torch.float64)


def check(net, X, Y, expected_loss, params):
    """Forward loss must match; hand-written grads must match autograd run on your own forward."""
    with torch.no_grad():
        loss = net.forward(X, Y)
        assert torch.isclose(loss, torch.tensor(expected_loss), atol=1e-8), \
            f"forward: loss {loss.item():.10f}, expected {expected_loss:.10f}"
        grads = net.backward()
    for name in params:
        getattr(net, name).requires_grad_(True)
    net.forward(X, Y).backward()
    for name in params:
        P = getattr(net, name)
        g = grads[name]
        assert g.shape == P.shape, f"{name}: grad shape {tuple(g.shape)} != param shape {tuple(P.shape)}"
        err = (g - P.grad).abs().max().item()
        assert torch.allclose(g, P.grad, atol=1e-10), f"{name}: max abs error {err:.2e}"
        print(f"{name}: max abs error {err:.1e}  ok")
        P.requires_grad_(False)
        P.grad = None


def data(seed, B=6, D=5, O=3, classes=False):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(B, D, generator=g)                                       # [B, D]
    Y = torch.randint(O, (B,), generator=g) if classes else torch.randn(B, O, generator=g)   # [B] or [B, O]
    return X, Y

## Problem 1: product of a layer with its own row sum (warm-up)

`H2 = H1_scaled * S` where `S` is built from `H1_scaled` itself: the gradient reaches `H1_scaled` along two paths.

In [ ]:
class GateNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = torch.randn(input_dim, hidden_dim) / input_dim ** 0.5
        self.W2 = torch.randn(hidden_dim, output_dim) / hidden_dim ** 0.5
        self.gamma = torch.randn(hidden_dim)

    def forward(self, X, Y_target):
        """
        1. Z1 = X @ W1                                   [B, H]
        2. H1 = tanh(Z1)                                 [B, H]
        3. H1_scaled = H1 * gamma                        [B, H]
        4. S = sum(H1_scaled, dim=1, keepdim=True)       [B, 1]
        5. H2 = H1_scaled * S                            [B, H]
        6. logits = H2 @ W2                              [B, C]
        7. log-softmax (stable), cross-entropy with integer labels Y_target [B], mean over the batch
        """
        # TODO
        raise NotImplementedError

    def backward(self):
        """Return {"W1": ..., "W2": ..., "gamma": ...}"""
        # TODO
        raise NotImplementedError

In [ ]:
torch.manual_seed(0)
net = GateNet(5, 8, 3)
X, Y = data(1, classes=True)
check(net, X, Y, 0.9131177361, ["W1", "W2", "gamma"])
print("ok")

## Problem 2: batch norm, then softmax cross-entropy

The classic. `mu` and `var` are computed over the batch, so every row's gradient depends on every other row.

In [ ]:
class BatchNormNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = torch.randn(input_dim, hidden_dim) / input_dim ** 0.5
        self.W2 = torch.randn(hidden_dim, output_dim) / hidden_dim ** 0.5
        self.gamma = torch.randn(hidden_dim)
        self.beta = torch.randn(hidden_dim)
        self.eps = 1e-5

    def forward(self, X, Y_target):
        """
        1. Z1 = X @ W1                                   [B, H]
        2. H1 = softplus(Z1) = log(1 + exp(Z1))          [B, H]
        3. mu = mean(H1, dim=0)                          [H]
        4. Hc = H1 - mu                                  [B, H]
        5. var = mean(Hc^2, dim=0)                       [H]
        6. Hn = Hc / sqrt(var + eps)                     [B, H]
        7. H2 = Hn * gamma + beta                        [B, H]
        8. logits = H2 @ W2                              [B, C]
        9. log-softmax (stable), cross-entropy with integer labels Y_target [B], mean over the batch
        """
        # TODO
        raise NotImplementedError

    def backward(self):
        """Return {"W1": ..., "W2": ..., "gamma": ..., "beta": ...}"""
        # TODO
        raise NotImplementedError

In [ ]:
torch.manual_seed(0)
net = BatchNormNet(5, 8, 3)
X, Y = data(1, classes=True)
check(net, X, Y, 1.2860775903, ["W1", "W2", "gamma", "beta"])
print("ok")

## Problem 3: max normalization, SiLU and a residual

`max` sends its gradient to the argmax only. The residual means `Hn` gets gradient from two places.

In [ ]:
class MaxResNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = torch.randn(input_dim, hidden_dim) / input_dim ** 0.5
        self.W2 = torch.randn(hidden_dim, hidden_dim) / hidden_dim ** 0.5
        self.W3 = torch.randn(hidden_dim, output_dim) / hidden_dim ** 0.5
        self.alpha = torch.randn(hidden_dim)

    def forward(self, X, Y_target):
        """
        1. Z1 = X @ W1                                   [B, H]
        2. H1 = relu(Z1)                                 [B, H]
        3. M = max(H1, dim=1, keepdim=True)              [B, 1]
        4. Hn = H1 / (M + 1e-6)                          [B, H]
        5. Z2 = Hn @ W2                                  [B, H]
        6. H2 = Z2 * sigmoid(Z2)                         [B, H]  (SiLU)
        7. H3 = H2 * alpha + Hn                          [B, H]  (residual)
        8. logits = H3 @ W3                              [B, C]
        9. log-softmax (stable), cross-entropy with integer labels Y_target [B], mean over the batch
        """
        # TODO
        raise NotImplementedError

    def backward(self):
        """Return {"W1": ..., "W2": ..., "W3": ..., "alpha": ...}"""
        # TODO
        raise NotImplementedError

In [ ]:
torch.manual_seed(0)
net = MaxResNet(5, 8, 3)
X, Y = data(1, classes=True)
check(net, X, Y, 0.9851048788, ["W1", "W2", "W3", "alpha"])
print("ok")

## Problem 4: fix the backward pass

The forward is correct. The backward has three bugs. Find and fix them; the check tells you which gradient is wrong.

In [ ]:
class DebugNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = torch.randn(input_dim, hidden_dim) / input_dim ** 0.5
        self.W2 = torch.randn(hidden_dim, output_dim) / hidden_dim ** 0.5
        self.beta = torch.randn(hidden_dim)

    def forward(self, X, Y_target):
        """
        (THIS IS CORRECT)
        1. Z1 = X @ W1                                   [B, H]
        2. H1 = sigmoid(Z1)                              [B, H]
        3. Hs = H1 + beta                                [B, H]
        4. H2 = Hs - mean(Hs, dim=1, keepdim=True)       [B, H]  (center each row)
        5. logits = H2 @ W2                              [B, C]
        6. log-softmax (stable), cross-entropy with integer labels Y_target [B], mean over the batch
        """
        self.X, self.Y = X, Y_target
        self.Z1 = X @ self.W1
        self.H1 = torch.sigmoid(self.Z1)
        self.Hs = self.H1 + self.beta
        self.H2 = self.Hs - self.Hs.mean(dim=1, keepdim=True)
        logits = self.H2 @ self.W2
        logp = logits - torch.logsumexp(logits, dim=1, keepdim=True)
        self.p = logp.exp()
        return -logp[torch.arange(X.shape[0]), Y_target].mean()

    def backward(self):
        B = self.X.shape[0]
        dlogits = self.p.clone()
        dlogits[torch.arange(B), self.Y] -= 1
        dW2 = self.H2.T @ dlogits
        dH2 = dlogits @ self.W2.T
        dHs = dH2
        dbeta = dHs.sum(dim=0)
        dZ1 = dHs * self.H1
        dW1 = self.X.T @ dZ1
        return {"W1": dW1, "W2": dW2, "beta": dbeta}

In [ ]:
torch.manual_seed(0)
net = DebugNet(5, 8, 3)
X, Y = data(1, classes=True)
check(net, X, Y, 1.3861306022, ["W1", "W2", "beta"])
print("ok")